# EDA Titanic — cuaderno 02

Análisis exploratorio sobre **Bronze** (datos tal como llegan) y **Silver** (datos limpios).
Correr de arriba abajo (*Restart & Run All*). No contiene lógica productiva: todo cálculo reutilizable vive en `flows/`. Conclusiones escritas en `reports/conclusiones_eda.md`.

## 1. Diagnóstico sobre Bronze

In [ ]:
import pandas as pd                 # tablas (DataFrame)
import numpy as np                  # cálculo numérico y condiciones vectorizadas
import matplotlib.pyplot as plt     # gráficos base
import seaborn as sns               # gráficos estadísticos de alto nivel
from scipy import stats             # pruebas estadísticas

from flows.common.paths import BRONZE, SILVER, GOLD, BRONZE_LABELED_NAME, SILVER_NAME

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13

df = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)

### Paso 1. ¿Cuántos pasajeros y variables hay?

In [ ]:
df.shape   # (filas, columnas): 891 pasajeros, 12 variables

### Paso 2. Variables, tipos y valores no nulos

In [ ]:
df.info()

### Paso 3. Primeras 10 filas

In [ ]:
df.head(10)

### Paso 4. Estadísticas descriptivas

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(exclude='number').describe()   # mirar 'unique' y 'top'

### Paso 5. Datos faltantes: tabla y mapa de calor

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Datos faltantes en Bronze (Age, Cabin y Embarked)')
plt.show()

## 2. Conociendo a los pasajeros (datos de Silver)

Desde aquí el cuaderno lee Silver, que ya trae las variables derivadas del módulo `flows/01_data_engineering.py` (regenerar con `make silver` si cambia).

In [ ]:
df = pd.read_parquet(SILVER / SILVER_NAME)
df.shape

### Paso 6. Género y clase

In [ ]:
sns.countplot(data=df, x='Sex')
plt.title('Pasajeros por género')
plt.show()

In [ ]:
sns.countplot(data=df, x='Sex', hue='Pclass')
plt.title('Género por clase')
plt.show()

In [ ]:
pd.crosstab(df['Sex'], df['Pclass'], normalize='columns')   # proporción por clase

### Paso 7. Edades (solo conocidas: Silver aún no imputa)

In [ ]:
sns.histplot(df['Age'].dropna(), bins=30)
plt.title('Distribución de edades (sin imputar)')
plt.show()
print('media:', df['Age'].mean().round(1), '| mediana:', df['Age'].median(),
      '| min:', df['Age'].min(), '| max:', df['Age'].max())

### Paso 8. Person (hombre, mujer o niño)

In [ ]:
df['Person'].value_counts()

In [ ]:
# Reto 8.1: cuántos 'child' podrían faltar entre los pasajeros sin edad registrada.
n_sin_edad = df['AgeOriginal'].isna().sum()          # la imputación no inventa niños
conocidos = df.loc[df['AgeOriginal'].notna(), 'Person']
prop_child = (conocidos == 'child').mean()
print(f'Sin edad: {n_sin_edad}; niños adicionales estimados: {n_sin_edad * prop_child:.0f}')

### Paso 9. Niños por clase

In [ ]:
pd.crosstab(df['Pclass'], df['Person'])

### Paso 10. Densidades (KDE) de la edad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
sns.kdeplot(data=df, x='Age', hue='Sex', fill=True, common_norm=False, ax=axes[0])
sns.kdeplot(data=df, x='Age', hue='Person', fill=True, common_norm=False, ax=axes[1])
sns.kdeplot(data=df, x='Age', hue='Pclass', fill=True, common_norm=False,
            palette='viridis', ax=axes[2])
axes[0].set_title('Por género'); axes[1].set_title('Por Person'); axes[2].set_title('Por clase')
plt.tight_layout(); plt.show()

## 3. Ubicación, embarque y familia

### Paso 11. Cubierta (letra de la cabina)

In [ ]:
df['TCabin'].value_counts()

### Paso 12. Conteo por cubierta (solo conocidas)

In [ ]:
cubierta = df.loc[df['TCabin'] != 'X', 'TCabin']
sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Pasajeros por cubierta (cabina registrada)')
plt.show()

In [ ]:
# La cubierta 'T' tiene un solo registro: anómala para estadística, no un dato erróneo.
# Se excluye SOLO del gráfico; la fila se conserva en los datos.
cubierta = cubierta[cubierta != 'T']
sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Cubiertas sin la anómala T')
plt.show()

### Paso 13. Puerto de embarque y clase

In [ ]:
sns.countplot(data=df, x='Embarked', hue='Pclass')
plt.title('Embarque por clase')
plt.show()

In [ ]:
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index')   # composición de cada puerto

### Pasos 14-15. Viaja solo o con familia

In [ ]:
df['Alone'].value_counts(normalize=True)

In [ ]:
sns.countplot(data=df, x='Alone')
plt.title('¿Viajaba solo?')
plt.show()

## 4. Supervivencia: el núcleo del EDA

### Paso 16. Panorama

In [ ]:
sns.countplot(data=df, x='Survivor')
plt.title('¿Sobrevivió?')
plt.show()
print('Tasa general de supervivencia:', round(df['Survived'].mean(), 3))

### Paso 17. Supervivencia por clase

In [ ]:
sns.catplot(data=df, x='Pclass', y='Survived', kind='point', height=4, aspect=1.4)
plt.title('Tasa de supervivencia por clase (barras = IC 95%)')
plt.ylabel('Tasa de supervivencia')
plt.show()

### Paso 18. Clase y tipo de persona

In [ ]:
sns.catplot(data=df, x='Pclass', y='Survived', kind='point', height=4, aspect=1.4,
            hue='Person', hue_order=['child', 'female', 'male'])
plt.ylabel('Tasa de supervivencia')
plt.show()

### Paso 19. Edad y supervivencia

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived')
plt.show()

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass')
plt.show()

In [ ]:
generaciones = [10, 20, 40, 60, 80]
sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass', x_bins=generaciones)
plt.show()

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived', hue='Sex', x_bins=generaciones)
plt.show()

### Paso 20. Tarifa y edad

In [ ]:
plt.scatter(df['Age'], df['Fare'], alpha=0.5)
plt.xlabel('Edad'); plt.ylabel('Tarifa')
plt.title('Tarifa vs. edad')
plt.show()

In [ ]:
sns.scatterplot(data=df, x='Age', y='Fare', hue='Pclass', alpha=0.5, palette='viridis')
plt.title('Tarifa vs. edad por clase')
plt.show()

## 5. Títulos del nombre (Pasos 25-26)

In [ ]:
df['TTitulo'].value_counts()

In [ ]:
pd.crosstab(df['TTitulo'], df['Sex'])

In [ ]:
# Tras consolidar: Master -> Mr, títulos raros -> Otro.
df['TTitulo'].value_counts(), df['NTitulo'].value_counts().sort_index()

## 6. Correlaciones, grupos y significancia (Pasos 27-29)

In [ ]:
cols = ['Survived', 'Pclass', 'NSex', 'Age', 'SibSp', 'Parch', 'Fare',
        'NEmbarked', 'NCabin', 'NTitulo']
corr = df[cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Matriz de correlación (lineal, no causal)')
plt.show()
corr['Survived'].sort_values()

### Tablas Gold y chi-cuadrado (regenerar con `make gold`)

In [ ]:
pd.read_parquet(GOLD / 'survival_by_class_sex.parquet')

In [ ]:
pd.read_parquet(GOLD / 'survival_by_embarked.parquet')

In [ ]:
pd.read_parquet(GOLD / 'survival_by_embarked_class.parquet')

In [ ]:
pd.read_parquet(GOLD / 'survival_by_fare_quartile_class.parquet')

In [ ]:
pd.read_parquet(GOLD / 'survival_by_family.parquet')

In [ ]:
pd.read_parquet(GOLD / 'outlier_summary.parquet')

In [ ]:
import json
json.loads((GOLD / 'chi2_sex_survival.json').read_text())

## 7. Visualizaciones avanzadas (Pasos 30-31)

In [ ]:
# Con 'AgeOriginal' se ven solo edades reales (sin la imputación).
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.boxplot(data=df, x='Sex', y='AgeOriginal', ax=axes[0, 0])
sns.boxplot(data=df, x='Pclass', y='AgeOriginal', ax=axes[0, 1])
sns.boxplot(data=df, x='Embarked', y='AgeOriginal', ax=axes[1, 0])
sns.boxplot(data=df, x='Survived', y='AgeOriginal', ax=axes[1, 1])
plt.tight_layout(); plt.show()

In [ ]:
sns.histplot(df['Fare'], kde=True, bins=40)
plt.title('Distribución de la tarifa')
plt.show()
print('media:', df['Fare'].mean().round(1), '| mediana:', df['Fare'].median(),
      '| asimetría (skew):', df['Fare'].skew().round(2))

## Conclusiones

Los hallazgos, las decisiones de limpieza y sus sesgos, y lo que NO podemos afirmar están en `reports/conclusiones_eda.md`.